# 03.04 — Vision : audit du CNN appris par PPO (sans segmentation supervisée)

Le CNN **est entraîné dans le notebook PPO 03.02**, conjointement avec le pilotage, sans aucune annotation. Ce notebook examine les images et la structure visuelle du modèle déjà entraîné; il ne prétend pas produire de masques U-Net de classes sémantiques.

In [ ]:
from pathlib import Path
import sys
CWD=Path.cwd().resolve()
REPO=next((p for p in [CWD,*CWD.parents] if (p/'assetto_corsa_gym'/'assetto_corsa_gym'/'AssettoCorsaPlugin').is_dir()),None)
if REPO is None:
    raise FileNotFoundError('Ouvrir la racine IA_projet_Ouarti_ds_github ou un sous-dossier dans VS Code.')
PROJECT=REPO/'Code'
GYM_ROOT=REPO/'assetto_corsa_gym'/'assetto_corsa_gym'
assert PROJECT.is_dir(), 'Dossier Code introuvable'
if sys.version_info[:2] != (3,9):
    raise RuntimeError('Selectionner le kernel Python 3.9.13 dans VS Code. Actuel : '+sys.version)
for name in ['datasets','models','reports']:
    (PROJECT/name).mkdir(parents=True, exist_ok=True)
print('Depot :',REPO,'\nCode :',PROJECT,'\nPython :',sys.executable)

In [ ]:
import numpy as np, cv2, matplotlib.pyplot as plt
from PIL import Image
files=sorted((PROJECT/'datasets').glob('session_*/images/*.png'))
print('Images disponibles :',len(files))
if files:
    img=np.asarray(Image.open(files[-1]).convert('L'))
    reduced=cv2.resize(img,(84,84),interpolation=cv2.INTER_AREA)
    fig,ax=plt.subplots(1,2,figsize=(10,4))
    ax[0].imshow(img,cmap='gray');ax[0].set_title('Capture brute')
    ax[1].imshow(reduced,cmap='gray');ax[1].set_title('Observation du PPO 84x84')
    for a in ax:a.axis('off')
    plt.show()
else:
    print('Capturer des images avec 02_Donnees/04_Capturer_Images.ipynb')

In [ ]:
from stable_baselines3 import PPO
from pathlib import Path
MODELE=PROJECT/'models'/'ppo_vision_telemetrie.zip'
if MODELE.is_file():
    agent=PPO.load(str(MODELE),device='cpu')
    print('Politique :',agent.policy.__class__.__name__)
    print('Espace image :',agent.observation_space['image'])
    print('Espace etat :',agent.observation_space['state'])
    print('Extracteur de caracteristiques :',agent.policy.features_extractor)
else:
    print('Aucun modele entrainne. La vision sera apprise dans 03.02.')

## U-Net : point scientifique

Un CNN entraîné par PPO découvre des **caractéristiques utiles pour conduire**, pas une segmentation des pixels avec étiquettes explicites. Pour répondre au cahier des charges « identifier chaque partie du circuit », il faut **en plus** un jeu de masques étiquetés (éventuellement généré automatiquement par un outil de segmentation du simulateur à développer) et un apprentissage dédié de U-Net. Ce notebook ne fabrique pas de masques fictifs.